<a href="https://colab.research.google.com/github/The-Curious-Cipher/LLM_all/blob/main/CRAN_RAG_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install rank_bm25
!pip install rdflib
!pip install faiss-cpu
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.9/124.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.1/417.1 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.45
    Uninstalling langchain-core-0.3.45:
      Successfully uninstalled langchain-core-0.3.45


In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
from rdflib import Graph, Literal
import os
from google.colab import userdata
from langchain_groq import ChatGroq

In [13]:

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['HUGGINGFACEHUB_API_TOKEN'] = userdata.get('HF_TOKEN')

In [14]:
from rdflib import Graph
g = Graph()
g.parse("isobim_case_study.rdf", format="xml")
print(f"Loaded {len(g)} triples.")


Loaded 4729 triples.


In [15]:
from rdflib import Graph, Literal

def get_local_name(uri):
    """
    Returns the local name from a URI.
    Uses '#' if available; otherwise, splits on '/'.
    """
    uri = str(uri)
    if '#' in uri:
        return uri.split('#')[-1]
    elif '/' in uri:
        # Remove trailing slash if exists, then get last part
        return uri.rstrip('/').split('/')[-1]
    return uri



# Define a set of properties (local names) that are relevant to your questions
relevant_properties = {
    "type", "hasWidth", "hasHeight", "hasFinishDate",
    "hasSolution", "isLate", "hasName", "hasType",
    "hasDocumentType", "hasActor", "hasActivity"
}

facts = []  # List to hold meaningful textual facts

for s, p, o in g:
    subj = get_local_name(s)
    pred = get_local_name(p)

    # Check if the object is a literal or a resource
    if isinstance(o, Literal):
        obj = str(o)
    else:
        obj = get_local_name(o)

    if pred in relevant_properties or isinstance(o, Literal):
        fact = f"{subj} {pred} {obj}."
        facts.append(fact)

print("Extracted meaningful RDF facts", len(facts))



Extracted meaningful RDF facts 3201


In [16]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(facts, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)
print("Facts shape:", len(facts) )

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)

print("FAISS index built with", index.ntotal, "vectors.")

Embeddings shape: (3201, 384)
Facts shape: 3201
FAISS index built with 3201 vectors.


In [17]:

def retrieve_rdf_info(query, top_k=None):
    """
    Retrieves the top_k most similar RDF facts to the query.
    If top_k is not provided, retrieves all facts.
    Handles cases where fewer than top_k results are available.
    """
    if top_k is None or top_k > len(facts):
        top_k = len(facts)  # Ensure top_k does not exceed the number of stored facts

    # Convert query to embedding vector
    query_embedding = model.encode(query, convert_to_numpy=True)
    query_embedding = np.expand_dims(query_embedding, axis=0)  # Reshape for FAISS

    # Search the FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # Handle cases where fewer than top_k facts are retrieved
    retrieved_facts = []
    for i in range(min(top_k, len(indices[0]))):  # Avoid out-of-range error
        if indices[0][i] < len(facts):  # Ensure index is valid
            retrieved_facts.append(facts[indices[0][i]])

    # If nothing was retrieved, return a fallback message
    if not retrieved_facts:
        retrieved_facts = ["No relevant information was found for this query."]

    return retrieved_facts


In [18]:

def format_retrieved_facts(facts, query):
    """
    Converts raw RDF triples like 'IfcWindow320439378 type Windows'
    into a meaningful, structured summary.
    """
    entity_counts = {}  # Dictionary to store counts of different types

    for fact in facts:
        parts = fact.split(" ")  # Splitting each fact into components
        if len(parts) >= 3:
            entity_type = parts[-1]  # Extract entity type (last word in sentence)
            entity_counts[entity_type] = entity_counts.get(entity_type, 0) + 1

    # Generate a summary statement
    summary = []
    for entity, count in entity_counts.items():
        summary.append(f"There are {count} {entity} in the project.")

    return summary


In [19]:
query = " Give me the activities with a finish date before 1/1/2025. "
retrieved_context = retrieve_rdf_info(query, top_k=min(200, len(facts)))  # Reduce top_k for debugging


for fact in retrieved_context:
    print("-", fact)
print("🔹 Formated Retrieved Context:")
frf=format_retrieved_facts(retrieved_context, query)
for fact in frf:
    print("-", fact)


- Lot_01_:_Pose_échafaudages_bâtiment_GBAA has_finish_date 2022-12-09T00:00:00.
- Lot_01_:_Pose_échafaudages_bâtiment_GMP_ has_finish_date 2022-03-04T00:00:00.
- Lot_10_:_Peinture_bâtiment_GBAA has_finish_date 2023-05-19T00:00:00.
- Lot_03_:_Démolitions_et_modiﬁcations_baies_bâtiment_GBAA has_finish_date 2022-12-09T00:00:00.
- Lot_03_:_Démolitions_et_modiﬁcations_baies_bâtiment_GMP has_finish_date 2022-03-04T00:00:00.
- Lot_10_:_Peinture_bâtiment_GMP_ has_finish_date 2022-08-05T00:00:00.
- Lot_10_:_ITE_bâtiment_GBAA has_finish_date 2023-05-05T00:00:00.
- Lot_10_:_ITE_bâtiment_GMP_ has_finish_date 2022-07-22T00:00:00.
- Lot_01_:_Dépose_échafaudages_bâtiment_GBAA has_finish_date 2023-06-09T00:00:00.
- Lot_13_:_Pose_des_gaines_bâtiment_GBAA has_finish_date 2023-03-31T00:00:00.
- Lot_13_:_Pose_des_gaines_bâtiment_GMP_ has_finish_date 2022-06-17T00:00:00.
- Lot_09_:_Dépose_et_repose_de_portes_bâtiment_GBAA has_finish_date 2023-03-03T00:00:00.
- Lot_14_:_Alimentations_électriques_atelier_GMP

In [21]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
MODEL_NAME = "gemma2-9b-it"
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name=MODEL_NAME)

# Define a simple function to interact with the model
def ask_groq(question):
    response = llm.invoke(question)
    return response.content.strip() if response else "No response received."

In [46]:
def generate_answer_with_llm(prompt):
    """
    Generates an answer using the Groq LLM.
    """
    response = llm.invoke(prompt)
    return response.content.strip() if response else "No response from LLM."

def answer_question_rag_local(query, context):
    """
    Uses an optimized prompt structure for answering queries with retrieved RDF context.
    """
    # Format context before passing to LLM
    formatted_context = format_retrieved_facts(context, query)

    # Convert structured context into a string
    context_str = "\n".join(formatted_context[:25])

    # Construct a structured prompt
    prompt = f"""You are an expert assistant skilled in data extraction and analysis.
Use the following structured RDF knowledge to answer the question.

Context:
{context_str}
--------------------
Background:
{context[:100]}
---------------------
Question: {query}

Instructions:
- If the question is about counting, return only the total number in a single sentence.
- If the question is descriptive, summarize key facts relevant to the query.
- Avoid unrelated details and do not generate unnecessary conversation.

Answer:"""

    print("🔹 Final Prompt Sent to Groq LLM:")
    print(prompt)  # Debugging step

    return generate_answer_with_llm(prompt)

# Example usage:
query = "How many Administration documents documents are in the project?"
retrieved_context = retrieve_rdf_info(query, top_k=len(facts))
response = answer_question_rag_local(query, retrieved_context)
print("Generated Answer:")
print(response)


🔹 Final Prompt Sent to Groq LLM:
You are an expert assistant skilled in data extraction and analysis. 
Use the following structured RDF knowledge to answer the question.

Context:
There are 99 Administration_documents. in the project.
There are 289 Class. in the project.
There are 2 Project_management. in the project.
There are 114 /uploads/planning.pdf. in the project.
There are 2 Legal_documents. in the project.
There are 6 Project. in the project.
There are 2 Building_systems_design. in the project.
There are 1 budget.. in the project.
There are 1 Prelimenary_technical_project. in the project.
There are 50 As_built_project. in the project.
There are 1 Master_plan. in the project.
There are 652 NamedIndividual. in the project.
There are 1 work.. in the project.
There are 114 Installation_activity. in the project.
There are 3 Recommandation. in the project.
There are 1 PDF. in the project.
There are 8 Manual_and_operation_guide. in the project.
There are 58 ObjectProperty. in the proj